# 01. Mô hình Dummy làm mốc so sánh

Dummy là mốc so sánh cho Logistic Regression, Random Forest và MLP. Chiến lược cố định `most_frequent` luôn dự đoán lớp xuất hiện nhiều nhất trong Train cho mọi sinh viên. Mô hình này không cần tìm siêu tham số hoặc tiền xử lý đặc trưng.

Notebook sử dụng bộ chia chung trong `data/splits/splits.json` và các hàm tính chỉ số trong `common/metrics.py`. Mô hình học trên Train rồi được đánh giá trên Validation; giữ Test cho bước đánh giá cuối của dự án.

Chạy lần lượt tất cả các cell để lưu bảng tổng hợp, chỉ số từng lớp, ma trận nhầm lẫn và dự đoán vào `reports/tables/dummy/`, đồng thời lưu pipeline vào `artifacts/dummy_pipeline.joblib` cùng tệp JSON mô tả. Chọn kernel đã cài các thư viện trong requirements của dự án.


## 1. Đọc dữ liệu và kiểm tra bộ chia chung

Các cột trong CSV được ngăn cách bằng dấu chấm phẩy. Trước khi tính mã băm, chuẩn hóa ký tự xuống dòng CRLF thành LF theo đúng cách của `00_prepare_data.ipynb`. Chỉ số trong bộ chia là vị trí dòng của CSV gốc. Kiểm tra cả ba tập để phát hiện chỉ số không hợp lệ, dòng lặp, dòng nằm ở nhiều tập hoặc dòng bị bỏ sót.


In [1]:
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

# Tìm thư mục gốc từ vị trí chạy hiện tại để notebook vẫn chạy được trong thư mục con.
here = Path.cwd().resolve()
root = next((folder for folder in (here, *here.parents)
             if (folder / "common/preprocessing.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Hãy mở notebook trong thư mục dự án đã tải về.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from common.metrics import (
    confusion_frame, evaluate, per_class_frame, summary_frame,
)
from common.preprocessing import FEATURE_COLUMNS, LABELS, TARGET

data_path = root / "data/raw/data.csv"
split_path = root / "data/splits/splits.json"
# CSV của bộ dữ liệu này dùng dấu chấm phẩy để ngăn cách các cột.
data = pd.read_csv(data_path, sep=";", encoding="utf-8-sig")
data.columns = data.columns.str.strip()
# Chuẩn hóa xuống dòng trước khi băm để kiểm tra đúng phiên bản dữ liệu đã chia.
csv_bytes = data_path.read_bytes().replace(b"\r\n", b"\n")
data_hash = hashlib.sha256(csv_bytes).hexdigest()
split = json.loads(split_path.read_text(encoding="utf-8"))
if split["sha256"] != data_hash or split["n_rows"] != len(data):
    raise ValueError("CSV không khớp với dữ liệu dùng để tạo bộ chia chung.")

# Đối chiếu tên cột và nhãn với quy ước dùng chung trước khi lấy Train.
required_columns = set(FEATURE_COLUMNS) | {TARGET}
missing_columns = required_columns - set(data.columns)
if missing_columns or data.columns.duplicated().any():
    raise ValueError(f"Các cột CSV không hợp lệ. Cột còn thiếu: {sorted(missing_columns)}")
if data[TARGET].isna().any() or not data[TARGET].isin(LABELS).all():
    raise ValueError("Target chứa nhãn bị thiếu hoặc nhãn không thuộc danh sách dùng chung.")

# Mỗi tập phải chứa chỉ số nguyên, nằm trong CSV và không lặp trong chính tập đó.
def checked_rows(name):
    rows = split[name]
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"{name} phải là danh sách chỉ số dòng và không được rỗng.")
    # Kiểm tra đúng kiểu int để không chấp nhận True/False như chỉ số dòng.
    if any(type(row) is not int or not 0 <= row < len(data) for row in rows):
        raise ValueError(f"{name} chứa chỉ số dòng không hợp lệ.")
    if len(set(rows)) != len(rows):
        raise ValueError(f"{name} chứa chỉ số dòng bị lặp.")
    return rows

train_rows = checked_rows("train")
validation_rows = checked_rows("validation")
test_rows = checked_rows("test")
# Ba tập phải tách biệt và ghép lại phải đủ mọi dòng của CSV gốc.
all_rows = train_rows + validation_rows + test_rows
if len(all_rows) != len(set(all_rows)):
    raise ValueError("Train, Validation và Test phải không có dòng nào trùng nhau.")
if set(all_rows) != set(range(len(data))):
    raise ValueError("Bộ chia chung phải bao gồm đầy đủ mọi dòng trong CSV.")

# X chỉ chứa đặc trưng; y là nhãn. Lát nữa dùng iloc để lấy đúng vị trí dòng đã lưu.
X = data.loc[:, list(FEATURE_COLUMNS)]
y = data[TARGET]
if set(y.iloc[train_rows]) != set(LABELS):
    raise ValueError("Train phải có đủ cả ba nhãn của dự án.")
print(f"CSV: {len(data):,} dòng; Train: {len(train_rows):,}; Validation: {len(validation_rows):,}")
# Chỉ thống kê tỷ lệ nhãn trên Train để hiểu Dummy sẽ chọn lớp nào.
train_counts = y.iloc[train_rows].value_counts().reindex(LABELS, fill_value=0)
display(pd.DataFrame({"count": train_counts, "proportion": train_counts / len(train_rows)}))

CSV: 4,424 dòng; Train: 3,096; Validation: 664


,count,proportion
Target,,
Dropout,995,0.321382
Enrolled,556,0.179587
Graduate,1545,0.499031


## 2. Huấn luyện mô hình mốc trên Train

Pipeline chỉ có một bước để các mô hình trong dự án có cùng cách lưu và nạp lại. Dummy bỏ qua giá trị đặc trưng và học tỷ lệ các lớp từ nhãn Train. Lớp có tỷ lệ cao nhất sẽ được dự đoán cho mọi dòng; nếu các lớp bằng nhau, mô hình chọn theo thứ tự lớp của bộ phân loại.


In [2]:
# Dummy bỏ qua giá trị X và chỉ học tỷ lệ các lớp từ y của Train.
model = Pipeline([("model", DummyClassifier(strategy="most_frequent"))])
model.fit(X.iloc[train_rows], y.iloc[train_rows])
classifier = model.named_steps["model"]
# Ghép vị trí tỷ lệ lớn nhất với đúng tên lớp trong classes_.
predicted_label = classifier.classes_[np.argmax(classifier.class_prior_)]
print(f"Lớp được dự đoán cho mọi dòng: {predicted_label}")
print(f"Tỷ lệ lớp này trong Train: {classifier.class_prior_.max():.2%}")

Lớp được dự đoán cho mọi dòng: Graduate
Tỷ lệ lớp này trong Train: 49.90%


## 3. Đánh giá trên Validation

Macro-F1 tính trung bình F1 của ba lớp với trọng số bằng nhau. Dummy chỉ dự đoán lớp đông nhất nên recall của hai lớp còn lại bằng 0, dù accuracy có thể nhìn khá cao. Các hàm tính chỉ số dùng chung quy ước precision bằng 0 khi một lớp không được dự đoán lần nào.

Với `most_frequent`, `predict_proba` trả về 1 cho lớp đông nhất và 0 cho các lớp còn lại. Đây là đầu ra cố định của mô hình mốc, không phải xác suất rủi ro đã được hiệu chỉnh cho từng sinh viên.


In [3]:
# Dùng bộ chia đã lưu; không tạo lại Validation hoặc sử dụng Test để chấm ở đây.
X_val = X.iloc[validation_rows]
y_val = y.iloc[validation_rows]
y_pred = model.predict(X_val)
# Xác suất trả về theo classes_; sắp lại theo LABELS để mọi mô hình cùng thứ tự lớp.
probabilities = pd.DataFrame(
    model.predict_proba(X_val), columns=model.classes_, index=X_val.index,
).reindex(columns=LABELS)
# Dùng cùng hàm metric với các mô hình khác để kết quả so sánh nhất quán.
result = evaluate(y_val, y_pred)
summary = summary_frame({"Dummy (most_frequent)": result})
per_class = per_class_frame(result)
confusion = confusion_frame(result)

display(summary)
display(per_class)
display(confusion)
print(f"Số dòng Validation: {result['n_samples']:,}")

,accuracy,macro_f1,dropout_recall,dropout_precision,dropout_f2
model,,,,,
Dummy (most_frequent),0.5,0.222222,0.0,0.0,0.0


,precision,recall,f1,f2,support
Dropout,0.0,0.0,0.000000,0.000000,213
Enrolled,0.0,0.0,0.000000,0.000000,119
Graduate,0.5,1.0,0.666667,0.833333,332


dự đoán,Dropout,Enrolled,Graduate
thực tế,,,
Dropout,0,0,213
Enrolled,0,0,119
Graduate,0,0,332


Số dòng Validation: 664


## 4. Lưu báo cáo và pipeline đã huấn luyện

Bảng dự đoán giữ chỉ số dòng của CSV gốc để có thể tra lại từng sinh viên. Các cột xác suất được sắp theo thứ tự nhãn dùng chung. Tệp metadata ghi danh sách đặc trưng đầu vào và tỷ lệ lớp học từ Train. Sau khi lưu, nạp lại pipeline và kiểm tra dự đoán Validation có khớp với trước khi lưu hay không.


In [4]:
table_dir = root / "reports/tables/dummy"
artifact_dir = root / "artifacts"
table_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

# Lưu riêng bảng tổng hợp, chỉ số từng lớp và ma trận nhầm lẫn để đọc và làm báo cáo.
summary.to_csv(table_dir / "validation_summary.csv")
per_class.to_csv(table_dir / "validation_per_class.csv")
confusion.to_csv(table_dir / "validation_confusion.csv")
# Giữ nguyên thứ tự Validation và chỉ số CSV gốc để tra lại các dự đoán sai.
predictions = pd.DataFrame({
    "row_index": validation_rows,
    "y_true": y_val.to_numpy(),
    "y_pred": y_pred,
})
# Ghi rõ tên lớp trong từng cột xác suất để tránh nhầm thứ tự khi đọc kết quả.
for label in LABELS:
    predictions[f"probability_{label}"] = probabilities[label].to_numpy()
predictions.to_csv(table_dir / "predictions_val.csv", index=False)

pipeline_path = artifact_dir / "dummy_pipeline.joblib"
joblib.dump(model, pipeline_path, compress=3)
# Nạp thử tệp vừa lưu để chắc chắn pipeline tái tạo đúng dự đoán Validation.
reloaded = joblib.load(pipeline_path)
if not np.array_equal(reloaded.predict(X_val), y_pred):
    raise ValueError("Dự đoán của pipeline sau khi nạp lại không khớp với trước khi lưu.")

# Ghi thông tin dữ liệu, bộ chia, đặc trưng và kết quả để nhóm kiểm tra lại lần chạy.
metadata = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "predicted_class": str(predicted_label),
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "notebooks/models/dummy/01_dummy_classifier.ipynb",
    "sklearn_version": sklearn.__version__,
    "data_sha256": data_hash,
    "split_seed": split["seed"],
    "labels": list(LABELS),
    "feature_columns": list(FEATURE_COLUMNS),
    "n_features_in": len(FEATURE_COLUMNS),
    "training_class_priors": {str(label): float(prior) for label, prior in
                             zip(classifier.classes_, classifier.class_prior_)},
    "n_train": len(train_rows),
    "n_validation": len(validation_rows),
    "validation_metrics": {key: float(summary.iloc[0][key]) for key in summary.columns},
}
(artifact_dir / "dummy_pipeline.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Đã lưu:")
for path in (table_dir, pipeline_path, artifact_dir / "dummy_pipeline.json"):
    print(f"- {path.relative_to(root)}")
display(summary)

Đã lưu:
- reports\tables\dummy
- artifacts\dummy_pipeline.joblib
- artifacts\dummy_pipeline.json


,accuracy,macro_f1,dropout_recall,dropout_precision,dropout_f2
model,,,,,
Dummy (most_frequent),0.5,0.222222,0.0,0.0,0.0


## Nhận xét và cách dùng kết quả

Dùng bảng tổng hợp Validation đã lưu làm mốc tối thiểu để so sánh với các mô hình còn lại. Mọi mô hình phải dùng cùng bộ chia và cùng hàm tính chỉ số, đặc biệt chú ý Macro-F1 và recall của lớp `Dropout`. Chiến lược Dummy đã được cố định trước, không được chọn dựa trên kết quả Validation hoặc Test.
